# 00b · Perfil estadístico agregado
Produce `perfil_lemas.json`, que contiene solo agregados (supresión de celdas con menos de 5 casos) y sirve para calibrar el dataset sintético del repositorio público. **Revise el JSON antes de compartirlo.**

In [ ]:
# @title 0 · Preparar el entorno (ejecutar primero)
# Si el repositorio ya está en GitHub, pegue su URL; si no, se pedirá subir el .zip.
REPO_URL = ""  # @param {type:"string"}

import importlib.util, os, subprocess, sys, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    RAIZ = Path("/content/prediccion-matricula-lemas")
    if not RAIZ.exists():
        if REPO_URL:
            subprocess.run(["git", "clone", "-q", REPO_URL, str(RAIZ)], check=True)
        else:
            from google.colab import files
            print("Suba el archivo prediccion-matricula-lemas.zip")
            nombre = next(iter(files.upload()))
            zipfile.ZipFile(nombre).extractall("/content")
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
REQUERIDOS = {'pandas': 'pandas'}  # módulo -> paquete pip
faltan = [pip for modulo, pip in REQUERIDOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Entorno:", "Google Colab" if EN_COLAB else "local", "| Raíz:", RAIZ)

In [ ]:
# @title 1 · Cargar base seudonimizada
import json, re, warnings
import pandas as pd
warnings.filterwarnings("ignore", message="Could not infer format")

# Marque SUBIR_DE_NUEVO si subió un archivo equivocado (se borra y se vuelve a pedir)
SUBIR_DE_NUEVO = False  # @param {type:"boolean"}
from src.utils import obtener_base_seud
ruta = RAIZ / "data" / "raw" / "base_seud.csv"
df = obtener_base_seud(ruta, EN_COLAB, SUBIR_DE_NUEVO)

sys.path.insert(0, str(RAIZ / "tools"))
from seudonimizar import normalizar, validar_cedula_ec
sospechosas = [c for c in df.columns if "nombre" in c.lower()]
sospechosas += [c for c in df.columns if len(df[c].dropna())
                and df[c].dropna().head(300).map(lambda v: validar_cedula_ec(normalizar(v) or "")).mean() > 0.5]
assert not sospechosas, f"❌ Identificadores detectados: {sospechosas}"
print(f"✅ {len(df)} filas × {df.shape[1]} columnas, sin identificadores directos")

In [ ]:
# @title 2 · Perfil por columna
from perfilar_datos import MIN_CELDA, perfilar_columna, _suprimir

PERFIL = {"nota": f"Agregados con supresión de celdas < {MIN_CELDA}. Sin filas individuales.",
          "tablas": {"base_seud": {"filas": int(len(df)),
                                   "columnas": {c: perfilar_columna(df[c], df["anoa"]) for c in df.columns}}}}
pd.DataFrame([(c, p["tipo"], p["pct_nulos"], p["n_unicos"])
              for c, p in PERFIL["tablas"]["base_seud"]["columnas"].items()],
             columns=["columna", "tipo", "% nulos", "únicos"])

In [ ]:
# @title 3 · Chequeos del proceso
CH = {}
f = pd.to_datetime(df["fecha_pago"], errors="coerce", format="mixed")
# La ventana se toma del año del ciclo (anoa), no del año de la fecha de pago
from src.data_processing import anio_de_ciclo
anio = anio_de_ciclo(df["anoa"]).astype("float")
t0 = pd.to_datetime(dict(year=anio, month=2, day=20), errors="coerce")
h = pd.to_datetime(dict(year=anio, month=4, day=30), errors="coerce")
ubic = pd.Series("sin fecha", index=df.index)
ubic[f < t0] = "antes de t0 (20-feb)"; ubic[(f >= t0) & (f <= h)] = "en plazo"; ubic[f > h] = "tardío"
# Antiguo = el estudiante (por seudónimo de cédula) estaba en la hoja del año anterior.
ids = {a: set(df.loc[anio == a, "id_seudonimo"]) for a in anio.dropna().unique()}
tipo = pd.Series("sin referencia", index=df.index)
for a in ids:
    if a - 1 in ids:
        filas = anio == a
        tipo[filas] = df.loc[filas, "id_seudonimo"].isin(ids[a - 1]).map({True: "antiguo", False: "nuevo"})
tabla = pd.crosstab([anio.fillna(-1).astype(int), tipo], ubic)
CH["pagos_por_anio_y_tipo"] = {f"{a} | {t}": _suprimir(r) for (a, t), r in tabla.iterrows()}
display(tabla)

tam = df.groupby(["anoa", "id_familia_seudonimo"]).size().clip(upper=4)
CH["estudiantes_por_representante"] = {f"{a} | {n}": (int(v) if v >= MIN_CELDA else f"<{MIN_CELDA}")
                                       for (a, n), v in tam.groupby(level=0).value_counts().sort_index().items()}
for col in ["Sede", "beca", "Reserva", "RColegio", "Tiempo", "P.Conducta"]:
    ct = pd.crosstab(df["anoa"], df[col].fillna("vacío").replace("", "vacío"))
    CH[f"anio_x_{col}"] = {str(a): _suprimir(r) for a, r in ct.iterrows()}
PERFIL["chequeos"] = CH
print("Chequeos:", list(CH))

In [ ]:
# @title 4 · Guardar y descargar
salida = RAIZ / "perfil_lemas.json"
salida.write_text(json.dumps(PERFIL, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"✅ {salida} ({salida.stat().st_size / 1024:.1f} KB). Revíselo antes de compartirlo.")
if EN_COLAB:
    from google.colab import files
    files.download(str(salida))